In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. CARREGAMENTO DOS DADOS E BUSCA SEGURA DAS COLUNAS
# ==============================================================================
file_path = 'TELESAPFormulrioMdic-EXPORTAOBOLSISTAAtua_DATA_LABELS_2025-07-23_1329.csv'
try:
    df = pd.read_csv(file_path, low_memory=False)
except FileNotFoundError:
    print(f"Erro: O arquivo {file_path} não foi encontrado.")
    raise

# Busca blindada: procura por 'raça' ou 'etnia'
colunas_raca = [c for c in df.columns if 'raça' in c.lower() or 'raca' in c.lower() or 'etnia' in c.lower()]

if not colunas_raca:
    print("ERRO: Nenhuma coluna de Raça/Etnia foi encontrada no arquivo CSV.")
else:
    col_raca = colunas_raca[0]

    cols_inf = [c for c in df.columns if 'doenças infecciosas' in c.lower() and 'choice=' in c.lower() and 'não' not in c.lower() and 'outras' not in c.lower()]
    cols_cro = [c for c in df.columns if 'doenças crônicas' in c.lower() and 'choice=' in c.lower() and 'não' not in c.lower() and 'outras' not in c.lower()]
    cols_ciap = [c for c in df.columns if 'ciap' in c.lower()]

    # ==============================================================================
    # 2. TRATAMENTO LONGITUDINAL (PROPAGAÇÃO) E MAPEAMENTO IBGE
    # ==============================================================================
    df = df.sort_values(['Record ID', 'Repeat Instance'], na_position='first')
    cols_to_fill = [col_raca] + cols_inf + cols_cro
    cols_to_fill = [c for c in cols_to_fill if c in df.columns]
    
    df[cols_to_fill] = df.groupby('Record ID')[cols_to_fill].ffill()

    df_ev = df[df['Repeat Instrument'].notna()].copy()
    df_ev = df_ev.dropna(subset=[col_raca])

    def mapear_raca_flexivel(val):
        v = str(val).lower().strip()
        if 'branca' in v or 'branco' in v: return 'Branca'
        if 'preta' in v: return 'Preta'
        if 'parda' in v or 'pardo' in v: return 'Parda'
        if 'amarela' in v or 'amarelo' in v: return 'Amarela'
        if 'indígena' in v or 'indigena' in v: return 'Indígena'
        return None

    df_ev['Grupo_Raca'] = df_ev[col_raca].apply(mapear_raca_flexivel)
    df_plot = df_ev.dropna(subset=['Grupo_Raca']).copy()

    # ==============================================================================
    # 3. CÁLCULO SEPARADO DA CARGA DE DOENÇAS
    # ==============================================================================
    def to_bin(val):
        return 1 if str(val).lower().strip() in ['checked', '1', '1.0', 'sim', 'verdadeiro'] else 0

    for c in cols_inf + cols_cro:
        df_plot[c] = df_plot[c].apply(to_bin).astype(int)

    # Cargas separadas
    df_plot['Carga Infecciosa'] = df_plot[cols_inf].sum(axis=1)
    df_plot['Carga Crônica'] = df_plot[cols_cro].sum(axis=1)

    # ==============================================================================
    # 4. PREPARAÇÃO (MELTING) PARA COMPARAR OS TIPOS DE DOENÇA
    # ==============================================================================
    df_melted = df_plot.melt(
        id_vars=['Record ID', 'Grupo_Raca'],
        value_vars=['Carga Infecciosa', 'Carga Crônica'],
        var_name='Tipo de Doença',
        value_name='Quantidade de Doenças'
    )

    ordem_raca = [g for g in ['Branca', 'Parda', 'Preta', 'Amarela', 'Indígena'] if g in df_plot['Grupo_Raca'].unique()]

    # ==============================================================================
    # 5. GERAÇÃO E EXPORTAÇÃO DAS BASES (CSV) PARA O ARTIGO
    # ==============================================================================
    
    # Base 1: Carga de Doenças por Raça/Cor
    tabela_carga = df_melted.groupby(['Grupo_Raca', 'Tipo de Doença'])['Quantidade de Doenças'].agg(['mean', 'std', 'count']).reset_index()
    tabela_carga.columns = ['Raça/Cor (IBGE)', 'Perfil Clínico', 'Média de Doenças', 'Desvio Padrão', 'N (Amostra)']
    
    # Reordenando a tabela para seguir o padrão IBGE
    tabela_carga['Raça/Cor (IBGE)'] = pd.Categorical(tabela_carga['Raça/Cor (IBGE)'], categories=ordem_raca, ordered=True)
    tabela_carga = tabela_carga.sort_values(['Raça/Cor (IBGE)', 'Perfil Clínico'])
    
    tabela_carga.to_csv('base_carga_doencas_raca.csv', index=False, encoding='utf-8-sig')

    # Base 2: Relatório CIAP-2 por Raça/Cor
    dados_ciap_raca = []
    if cols_ciap:
        col_ciap_alvo = cols_ciap[0]
        for grupo in ordem_raca:
            sub = df_plot[df_plot['Grupo_Raca'] == grupo]
            top_ciaps = sub[col_ciap_alvo].dropna().value_counts().head(5) # Exporta o Top 5 para o CSV
            
            for ciap, contagem in top_ciaps.items():
                dados_ciap_raca.append({
                    'Raça/Cor (IBGE)': grupo,
                    'N Total (Grupo)': len(sub),
                    'Motivo CIAP-2': str(ciap).strip(),
                    'Frequência': contagem,
                    'Proporção (%)': round((contagem / len(sub) * 100), 1) if len(sub) > 0 else 0
                })
        
        tabela_ciap = pd.DataFrame(dados_ciap_raca)
        tabela_ciap.to_csv('base_ciap_por_raca.csv', index=False, encoding='utf-8-sig')

    print(f"\n{'='*80}")
    print("✅ BASES DE DADOS EXPORTADAS COM SUCESSO:")
    print("- 'base_carga_doencas_raca.csv' (Estatísticas do Gráfico)")
    print("- 'base_ciap_por_raca.csv' (Motivos de Consulta)")
    print(f"{'='*80}")

    # ==============================================================================
    # 6. RELATÓRIO NO TERMINAL (SÍNTESE NUMÉRICA E CIAP-2)
    # ==============================================================================
    print(f"\n{'='*80}")
    print("SÍNTESE NUMÉRICA: MÉDIA DE DOENÇAS POR GRUPO")
    print(f"{'='*80}")
    medias = df_plot.groupby('Grupo_Raca')[['Carga Infecciosa', 'Carga Crônica']].mean()
    print(medias.reindex(ordem_raca).round(3))

    print(f"\n{'='*80}")
    print("PRINCIPAIS MOTIVOS DE CONSULTA (CIAP-2) POR RAÇA/COR")
    print(f"{'='*80}")

    if not cols_ciap:
        print("Nenhuma coluna contendo o termo 'CIAP' foi encontrada.")
    else:
        for grupo in ordem_raca:
            sub = df_plot[df_plot['Grupo_Raca'] == grupo]
            print(f"\n[{grupo.upper()}] - Atendimentos analisados: {len(sub)}")
            if len(sub) > 0:
                top_ciaps = sub[col_ciap_alvo].dropna().value_counts().head(3)
                if len(top_ciaps) > 0:
                    for ciap, contagem in top_ciaps.items():
                        print(f"  -> {str(ciap)[:60]:<60} ({contagem} casos)")
                else:
                    print("  -> Sem CIAP registrado.")
    print(f"\n{'='*80}")

    # ==============================================================================
    # 7. VISUALIZAÇÃO: BARPLOT AGRUPADO (INFECCIOSAS VS CRÔNICAS)
    # ==============================================================================
    plt.figure(figsize=(12, 8))
    sns.set_theme(style="whitegrid")

    paleta_doencas = {'Carga Infecciosa': '#e74c3c', 'Carga Crônica': '#2980b9'}

    ax = sns.barplot(
        data=df_melted,
        x="Grupo_Raca",
        y="Quantidade de Doenças",
        hue="Tipo de Doença",
        palette=paleta_doencas,
        capsize=.05,
        err_kws={'linewidth': 1.5},
        order=ordem_raca
    )

    plt.title("Comparativo: Carga de Doenças Infecciosas vs. Crônicas por Raça/Cor", fontsize=16, fontweight='bold', pad=20)
    plt.ylabel("Média de Doenças por Paciente (com IC 95%)", fontsize=13, fontweight='bold')
    plt.xlabel("Raça / Cor (Padrão IBGE)", fontsize=13, fontweight='bold')

    sns.despine(left=True, bottom=True)
    plt.legend(title="Tipo de Perfil Clínico", bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False, fontsize=12)

    plt.tight_layout()
    plt.show()